# Prompt 03: Zero-Shot, Few-Shot, N-Shot

1. Sweep shot counts; measure accuracy vs cost
2. Order sensitivity test
3. Dynamic few-shot with embeddings
4. Negative-example demo
5. Exercises: per-feature shot-count tuning

Deps: `pip install anthropic openai sentence-transformers numpy`

## 1. Setup

In [ ]:
import os

def call_msgs(msgs, system='You classify tickets.'):
    if os.getenv('ANTHROPIC_API_KEY'):
        from anthropic import Anthropic
        r = Anthropic().messages.create(model='claude-sonnet-4-6', max_tokens=30, temperature=0, system=system, messages=msgs)
        return r.content[0].text.strip()
    if os.getenv('OPENAI_API_KEY'):
        from openai import OpenAI
        r = OpenAI().chat.completions.create(model='gpt-4o-mini', max_tokens=30, temperature=0,
            messages=[{'role':'system','content':system}]+msgs)
        return r.choices[0].message.content.strip()
    return '[no provider]'

## 2. Custom Classification Task with Labeled Pool

Classify internal tickets into `billing`, `account`, `bug`, or `feature-request`. The category names imply the task but the exact cutoffs benefit from examples.

In [ ]:
pool = [
    ('My invoice shows a charge I don\'t recognize.',             'billing'),
    ('I was double-charged for October.',                          'billing'),
    ('Please cancel my subscription for next cycle.',             'billing'),
    ('I can\'t log in; it says my password is wrong.',             'account'),
    ('Two-factor is stuck in a loop on my phone.',                 'account'),
    ('Change the email on file to alice@new.com.',                'account'),
    ('Export button returns a 500 error on large orgs.',           'bug'),
    ('Dashboard sidebar flickers on Safari.',                      'bug'),
    ('Save fails silently when title has emoji.',                  'bug'),
    ('Can you add dark mode?',                                     'feature-request'),
    ('Please support multi-currency reporting.',                   'feature-request'),
    ('Slack notifications when jobs finish would be helpful.',     'feature-request'),
]

eval_set = [
    ('My card expired and renewal failed.',                  'billing'),
    ('Magic link emails never arrive anymore.',              'account'),
    ('Export of 10M rows crashes the browser tab.',          'bug'),
    ('Add a bulk import via CSV.',                           'feature-request'),
    ('The invoice PDF has the wrong logo.',                  'bug'),
    ('I want to request a refund for last month.',           'billing'),
    ('Locked out after too many failed attempts.',           'account'),
    ('Please support SAML SSO.',                             'feature-request'),
]

## 3. Shot-Count Sweep

In [ ]:
def build_msgs(n_shot, query):
    msgs = []
    # rotate through classes so shots are balanced
    by_class = {}
    for text, label in pool:
        by_class.setdefault(label, []).append(text)
    picks = []
    while len(picks) < n_shot:
        for label, texts in by_class.items():
            if texts and len(picks) < n_shot:
                picks.append((texts.pop(0), label))
    for text, label in picks:
        msgs.append({'role':'user','content': f'Classify: {text}\nLabel only.'})
        msgs.append({'role':'assistant','content': label})
    msgs.append({'role':'user','content': f'Classify: {query}\nLabel only.'})
    return msgs

results = {}
for n in [0, 1, 2, 4, 8]:
    hits = 0
    for q, gold in eval_set:
        pred = call_msgs(build_msgs(n, q), system='You classify support tickets into exactly one of: billing, account, bug, feature-request.')
        hits += int(pred.strip().lower().startswith(gold))
    results[n] = hits / len(eval_set)
    print(f'n_shot={n}  accuracy={results[n]:.2f}')

## 4. Order Sensitivity

Fix n=4 shots. Try three orderings and measure how much accuracy varies.

In [ ]:
import random
random.seed(0)

def build_msgs_fixed(shots, query):
    msgs = []
    for text, label in shots:
        msgs.append({'role':'user','content': f'Classify: {text}'})
        msgs.append({'role':'assistant','content': label})
    msgs.append({'role':'user','content': f'Classify: {query}'})
    return msgs

fixed_shots = [(pool[0][0], pool[0][1]),
               (pool[3][0], pool[3][1]),
               (pool[6][0], pool[6][1]),
               (pool[9][0], pool[9][1])]

orders = {
    'canonical': list(fixed_shots),
    'reverse':   list(reversed(fixed_shots)),
    'shuffled':  random.sample(fixed_shots, len(fixed_shots)),
}

for name, shots in orders.items():
    hits = 0
    for q, gold in eval_set:
        pred = call_msgs(build_msgs_fixed(shots, q), system='Return exactly one of: billing, account, bug, feature-request.')
        hits += int(pred.strip().lower().startswith(gold))
    print(f'{name:<10} accuracy={hits/len(eval_set):.2f}')

## 5. Dynamic Few-Shot via Embeddings

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    import numpy as np
    emb_model = SentenceTransformer('all-MiniLM-L6-v2')

    pool_texts = [t for t, _ in pool]
    pool_emb = emb_model.encode(pool_texts, normalize_embeddings=True)

    def retrieve_shots(query, k=4):
        q = emb_model.encode([query], normalize_embeddings=True)
        sim = (q @ pool_emb.T)[0]
        top = np.argsort(-sim)[:k]
        # least-similar first, most-similar last (positional leverage)
        top = list(reversed(top.tolist()))
        return [pool[i] for i in top]

    hits = 0
    for q, gold in eval_set:
        shots = retrieve_shots(q, k=4)
        pred = call_msgs(build_msgs_fixed(shots, q), system='Return exactly one of: billing, account, bug, feature-request.')
        hits += int(pred.strip().lower().startswith(gold))
    print(f'dynamic few-shot (k=4): accuracy={hits/len(eval_set):.2f}')
except Exception as e:
    print('Skipped:', e)

## 6. Negative / Contrastive Examples

In [ ]:
# Task: classify benign vs phishing URL descriptions. Add contrastive examples.

system = 'Classify the message as SAFE or UNSAFE. Output only the label.'

pos_only = [
    {'role':'user','content':'Your invoice is available in your account.'},
    {'role':'assistant','content':'SAFE'},
    {'role':'user','content':'URGENT: click this link to avoid account suspension.'},
    {'role':'assistant','content':'UNSAFE'},
    {'role':'user','content':'Click here to reset your password to stop the hack.'},
]

print('--- positive + one negative ---')
print(call_msgs(pos_only, system=system))

contrastive = [
    {'role':'user','content':'Hi, please find attached our monthly newsletter.'},
    {'role':'assistant','content':'SAFE'},
    {'role':'user','content':'Your account will be closed unless you click the link now!'},
    {'role':'assistant','content':'UNSAFE'},
    {'role':'user','content':'A reminder that your package is on the way.'},
    {'role':'assistant','content':'SAFE'},
    {'role':'user','content':'Verify your credit card now or we will terminate service.'},
    {'role':'assistant','content':'UNSAFE'},
    {'role':'user','content':'Click here to reset your password to stop the hack.'},
]

print('\n--- contrastive few-shot ---')
print(call_msgs(contrastive, system=system))

## 7. Exercise: Per-Feature Shot-Count Tuning

Pick 3 features you care about. For each:
1. Build a 20-example eval set.
2. Sweep `n_shot ∈ {0, 1, 2, 4, 8}`.
3. Pick the lowest `n` where accuracy is within 2 points of the best.

That's your cost-optimal shot count. Document and lock it in.

## 8. Takeaways

- **Zero-shot first**; add examples only when measurement supports it.
- **Quality beats quantity.** 3 great shots > 10 mediocre.
- **Order matters** — put the most-similar example last.
- **Dynamic few-shot scales** with labeled example count.
- **Contrastive examples** teach decision boundaries efficiently.

Next: **04 — Chain-of-Thought & Reasoning Prompts** — teaching the model to think out loud.